# 數據清理與處理步驟報告

## 目錄
1. [專案概述]
2. [數據爬取與清理]
3. [數據合併與處理]
4. [最終結果與保存]

---

## 專案概述

本專案的目標是收集台灣上市公司的財報發布日期、損益表數據與股票歷史交易數據，進行數據清理與合併，最終生成一份包含股票交易與財報關聯的數據集，便於後續進行分析與建模。
由於在公開資訊觀測站抓取的綜合損益表沒有記載公開日期，所以在同個平台上找到各家公司上傳財報的日期當作財報發布日。

---

## 數據爬取與清理

### 1. 爬取財報發布日期

#### 步驟
1. 透過 公開資訊觀測站 網站爬取指定公司與年份的財報發布日期。
2. 使用多層次爬取方法，涵蓋一般公司與金融股特殊網頁結構。
3. 將日期轉換為標準的 `datetime64` 格式，並根據財報季節分配相應季度資訊。

#### 主要處理：
- **民國年份轉換**：轉為西元年份。
- **清除重複資料**：移除重複行，保證數據完整性。
- **存儲結果**：每家公司生成一份 CSV 文件，命名格式為 `PublishDate_{ticker}.TW.csv`。

---

### 2. 處理損益表數據

#### 步驟
1. 讀取綜合損益表數據。
2. 處理多版本損益表欄位，動態匹配收入、淨利、每股盈餘等指標。
3. 根據公司 ID 和年份，篩選相關季度數據。

#### 主要處理：
- **欄位兼容性處理**：解決不同行業或年份欄位名稱的變更。
- **數據清洗**：清除損益表中無關的數據行。
- **數據存儲**：整合後的數據存為 `modified_revenue.csv`。

---

## 數據合併與處理

### 1. 股票交易數據處理

#### 步驟
1. 讀取股票歷史交易數據。
2. 清除無效欄位（如Capital Gains、Stock Splits、Dividends等）。
3. 過濾成交量為零的交易日，通常是節假日。

#### 主要處理：
- **日期格式統一**：將日期欄位標準化為無時區的 `datetime` 格式。
- **欄位重命名**：確保公司代號與日期欄位名稱與財報數據一致。
- **數據排序**：按日期升序排列交易數據。

---

### 2. 合併數據

#### 步驟
1. 使用 `merge_asof` 方法，將財報數據與股票數據進行合併。
2. 基於公司代號和日期進行匹配，確保財報數據合併至下一個交易日。

#### 主要處理：
- **合併方向**：向前匹配（`direction='forward'`），避免財報數據與當日股票數據重疊。
- **排序與驗證**：合併後再次檢查數據排序及完整性。

---

## 最終結果與保存

### 數據儲存
- **輸出數據格式**：將最終整合的數據存為 `merged_financial_stock_data.csv`。
- **文件結構**：
  - 日期 (`date`)
  - 公司代號 (`company_id`)
  - 每日成交量 (`Volume`)
  - 每日價格數據（開盤價、收盤價等）
  - 財報數據（營業收入、淨利、每股盈餘等）


In [ ]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime
import os
from pathlib import Path
import time as Time
import twstock
import yfinance
from datetime import datetime
from bs4 import BeautifulSoup
import random
set_twstock = set(stock for stock in twstock.twse)
set_twstock = list(filter(lambda x: (len(x)<5 and x[:2]!="00"), sorted(set_twstock)))
"2330" in set_twstock


In [ ]:
# 爬取財報發布日期
def get_publish_date(ticker:str, year:int):
    if year > 1000:
        year -= 1911
    sess = requests.Session()
    url = "https://mops.twse.com.tw/mops/web/ajax_t57sb01_q1"
    payload = {
        "encodeURIComponent": "1",
        "step": "1",
        "firstin": "1",
        "off": "1",
        "TYPEK": "all",
        "keyword4":"" ,
        "code1": "",
        "TYPEK2": "",
        "checkbtn": "",
        "queryName": "co_id",
        "inpuType": "co_id",
        "co_id": ticker,
        "year": str(year),
    }
    headers = {
        "Cookie":"_cfuvid=ZVva1TY.q5e.xzvyrDwauRLUHSGjSQOKc1Pxy4w5wD0-1735181288434-0.0.1.1-604800000; _ga_J2HVMN6FVP=GS1.1.1735265061.4.0.1735265061.60.0.0; _gid=GA1.3.846327121.1736136400; _ga_5XRGGGWBYX=GS1.3.1736136400.1.0.1736136400.0.0.0; _ga=GA1.1.1747665681.1733886612; _ga_LTMT28749H=GS1.1.1736219719.17.1.1736219721.0.0.0",
        "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
        "content-type":"application/x-www-form-urlencoded"
    }
    try:
        # 進入網頁後爬取第二網址在進入
        response = sess.post(url=url, data=payload, headers=headers)
        html_parser = BeautifulSoup(response.text, "html.parser")
        second_url = html_parser.find("input")["value"].split("'")[1].strip("&")
        response = sess.get(url=second_url, headers=headers)
        html_parser = BeautifulSoup(response.text, "html.parser")
        df_list = pd.read_html(response.text)
        # 回傳有效表格
        for i in range(len(df_list)):
            if df_list[i].get("資料年度") is not None:
                return df_list[i]
        # 金融股財報會有第三層網頁
        third_url = "https://doc.twse.com.tw" +html_parser.find("form")["action"]  
        # 蒐集隱藏的input payload   
        payload_list = html_parser.find_all("input")
        payload_list = list(filter(lambda x: x["type"]=="hidden", payload_list))
        third_payload = { pl["name"] : pl["value"] for pl in payload_list }
        # 提交圖片的滑鼠點擊位置
        third_payload["x"] = str(random.randint(0,92))
        third_payload["y"] = str(random.randint(0,32))
        
        response = sess.post(url=third_url, data=third_payload, headers=headers)       
        df_list = pd.read_html(response.text)
        # 等待網頁回應
        Time.sleep(0.5)
        for i in range(len(df_list)):
            if df_list[i].get("資料年度") is not None:
                return df_list[i]        

    except Exception as e:
        print(ticker,year,e)



In [ ]:
# 上市股票財報發布日期爬蟲開始
curent_path = os.getcwd()     
base_path = Path(curent_path).resolve().parent
frame = {
    "Ticker":[],
    "Year":[],
    "Season":[],
    "Publish_date":[],
}
iO_counter = 0
sleep_time = 5

for tik, ticker in enumerate(set_twstock):
    for year in range(2000,2025):
        df = get_publish_date(ticker,year)
        iO_counter += 1
        print("IO count:",iO_counter)        
        for idx, season in enumerate(["第一季","第二季","第三季","第四季"]):
            try:
                # 找到每季的資料上傳日期，將日期轉換格式 datetime64
                upload = list(df[df["資料年度"].str.contains(season, na=False)].head(1)["上傳日期"])[0]
                date, time = upload.split(" ")
                d_year,month,day = map(int,date.split("/"))
                # 民國年轉西元年
                d_year += 1911
                hour, min, sec = map(int,time.split(":"))
                new_date = datetime(d_year,month,day,hour,min,sec)
                # 暫存於frame
                frame["Ticker"].append(ticker)
                frame["Year"].append(year)
                frame["Season"].append(idx+1)
                frame["Publish_date"].append(new_date)
            except Exception as e:
                print(ticker,year,season,e)
                continue
        sleep_time = 5 - ((tik+1) % 3)
        print("Sleep time:", sleep_time)
        Time.sleep(sleep_time)
    # 存成csv
    new_df = pd.DataFrame(frame)    
    path = f"{base_path}/財報發布日期/PublishDate_{ticker}.TW.csv"
    new_df.to_csv(path)
    frame = {
        "Ticker":[],
        "Year":[],
        "Season":[],
        "Publish_date":[],
    }
        

In [ ]:
# 開啟合併損益表，過濾數據
modified_dict = {
    "date":[],
    "company_id":[],
    "revenue":[],
    "net_profit":[],
    "EPS":[],
}
curent_path = os.getcwd()     
base_path = Path(curent_path).resolve().parent
path = f"{base_path}/綜合損益表/合併綜合損益表.csv"
revenue_df = pd.read_csv(path)
# loop 開啟每份日期，將需要的數據合併成一份表
def get_publish_date(ticker:str, book:dict, revenue_df:pd.DataFrame) -> dict:
    def chose_valid_col(candidates:list, series:dict) -> str:
        for candi in candidates:
            if not pd.isna(series[candi]):
                return candi
        return ""
    try:
        curent_path = os.getcwd()     
        base_path = Path(curent_path).resolve().parent
        path = f"{base_path}/財報發布日期/PublishDate_{ticker}.TW.csv"
        df = pd.read_csv(path)
        df = df.drop_duplicates()
        years = set(df["Year"])

        for y in years:
            siery = df[df["Year"]==y]
            
            for i in range(len(siery)):
                slc_siery = dict(siery.iloc[i])                      
                revenue_siery = revenue_df[(revenue_df["年度"] == int(y)) & (revenue_df["季度"] == "Q"+str(slc_siery.get("Season")))]
                if not revenue_siery.empty:
                    revenue_siery = dict(revenue_siery.iloc[0])
                else:
                    continue
                # 損益表欄目名稱一共換過3次，需要找出該年的各項數據寫在那些欄目中
                revenue = revenue_siery.get(chose_valid_col(["營業收入 淨額", "營業收入"], revenue_siery))
                net_profit = revenue_siery.get(chose_valid_col(["稅後純益", "本期淨利（淨損）","本期淨利(淨損)"], revenue_siery))                             
                EPs = revenue_siery.get(chose_valid_col(["每股稅後盈餘(元)", "基本每股盈餘","基本每股盈餘（元）"], revenue_siery))
                # 資料暫存在dict
                book["date"].append(slc_siery.get("Publish_date"))
                book["company_id"].append(ticker)
                book["revenue"].append(revenue), book["net_profit"].append(net_profit), book["EPS"].append(EPs)
                     
        return book
    except Exception as e:
        print(ticker,e)
# loop開始過濾company_id 傳入function，將所有數值儲存在modified_dict
for ticker in set_twstock:
    modified_dict = get_publish_date(ticker, modified_dict, revenue_df[revenue_df["公司代號"]==int(ticker)])

#儲存資料 
modified_DataFrame = pd.DataFrame(modified_dict)
modified_DataFrame.to_csv(f"{base_path}/整合/modified_revenue.csv",index=False)


In [ ]:
curent_path = os.getcwd()     
base_path = Path(curent_path).resolve().parent
financials_path =  f"{base_path}/綜合損益表/modified_revenue.csv"
stocks_path = f"{base_path}/上市股票歷史成交價/HistoryPriceFrom2000.csv"
# 讀取財報數據
financials = pd.read_csv(financials_path)
financials['date'] = pd.to_datetime(financials['date'])

# 讀取股票數據
stocks = pd.read_csv(stocks_path)
stocks['Date'] = pd.to_datetime(stocks['Date']).dt.tz_localize(None)

# 過濾無效欄位
del_unname = [col for col in stocks.columns if "Unnamed" in col][0]
stocks = stocks.drop(columns = [del_unname,"Dividends","Stock Splits","Capital Gains"])

# 確保日期格式統一
stocks = stocks.rename(columns={'Date': 'date', "Ticker":"company_id"})

# 將成交量為零的交易日剔除並排序
stocks = stocks[stocks["Volume"] != 0]
financials = financials.sort_values(by=['date'])
stocks = stocks.sort_values(by=['date'])


# 使用 merge_asof 合併，方向為 'forward'，確保財報數據合併到下一個交易日
merged = pd.merge_asof(
    stocks,
    financials,
    by='company_id',  # 基於公司 ID 合併
    on='date',        # 基於日期合併
    direction='forward'  # 向前合併到下一個交易日
)

# 將結果儲存
merged.to_csv('merged_financial_stock_data.csv', index=False)

